# Credit Card Customer Churn: Analysis and Prediction

A bank wants to know which credit card customers are about to leave, and why. This project explores 10,127 customers, finds the behaviours linked to churn, and builds models to flag at-risk customers early.

**Data:** Kaggle "Credit Card Customers" (BankChurners.csv). Target: `Attrition_Flag` (Attrited Customer = churned).

**Questions**
1. How many customers churn, and who are they?
2. Which behaviours (not just demographics) separate churners from loyal customers?
3. Can a model predict churn, and how well does it find the customers who really leave?
4. What should the bank do about it?

## 1. Setup and load data
In Colab, run the next cell and select `BankChurners.csv` when asked.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score,
                             roc_auc_score, average_precision_score, roc_curve,
                             precision_recall_curve, confusion_matrix)

sns.set_theme(style='whitegrid')
plt.rcParams['figure.dpi'] = 110
RANDOM_STATE = 42

In [ ]:
from google.colab import files
uploaded = files.upload()

In [ ]:
df = pd.read_csv('BankChurners.csv')
print(df.shape)
df.head()

## 2. Data quality checks

In [ ]:
print('Missing values:', df.isna().sum().sum())
print('Duplicate customer IDs:', df['CLIENTNUM'].duplicated().sum())
print()
for col in ['Education_Level', 'Marital_Status', 'Income_Category']:
    print(col, '-> Unknown:', (df[col] == 'Unknown').sum())

df['churn'] = (df['Attrition_Flag'] == 'Attrited Customer').astype(int)
print()
print(df['Attrition_Flag'].value_counts())
print(f"Churn rate: {df['churn'].mean():.1%}")

**Decisions**
- No missing values or duplicates. Some categories are labelled `Unknown`; I keep them as their own category instead of guessing.
- `CLIENTNUM` is just an ID, so it is not used as a feature.
- `Avg_Open_To_Buy` is basically Credit Limit minus Revolving Balance, so it is dropped from the models to avoid duplicate information.
- Only about 16% of customers churn (imbalanced), so accuracy alone is not enough. I focus on recall, precision, F1 and AUC for the churn class.

## 3. Who churns? Demographics

In [ ]:
fig, ax = plt.subplots(1, 4, figsize=(17, 3.8), sharey=True)
for a, col in zip(ax, ['Gender', 'Income_Category', 'Card_Category', 'Education_Level']):
    rate = df.groupby(col)['churn'].mean().sort_values(ascending=False) * 100
    sns.barplot(x=rate.index, y=rate.values, color='#1f6feb', ax=a)
    a.axhline(df['churn'].mean() * 100, color='red', ls='--', lw=1)
    a.set_title(f'Churn % by {col}')
    a.set_xlabel(''); a.tick_params(axis='x', rotation=45)
ax[0].set_ylabel('Churn rate (%)')
plt.tight_layout()
plt.savefig('01_churn_by_demographics.png', dpi=150, bbox_inches='tight')
plt.show()
print('Gender:\n', (df.groupby('Gender')['churn'].mean() * 100).round(1).to_string())
print('\nCard (customers):\n', df.groupby('Card_Category')['churn'].agg(['mean', 'count']).round(3).to_string())

Red dashed line = overall churn rate (16.1%). Demographics (age, gender, income, education) move the churn rate only a little. Behaviour matters much more, as the next charts show.

## 4. Behaviour: how churners differ

In [ ]:
df['trans_ct_band'] = pd.cut(df['Total_Trans_Ct'], [0, 40, 60, 80, 100, 200],
                             labels=['<=40', '41-60', '61-80', '81-100', '100+'])
df['zero_revolving'] = np.where(df['Total_Revolving_Bal'] == 0, 'Zero balance', 'Has balance')

fig, ax = plt.subplots(2, 2, figsize=(11, 8))
panels = [('Total_Relationship_Count', 'Products held with the bank'),
          ('Contacts_Count_12_mon', 'Bank contacts in last 12 months'),
          ('trans_ct_band', 'Transactions in last 12 months'),
          ('zero_revolving', 'Revolving balance')]
for a, (col, title) in zip(ax.flat, panels):
    rate = df.groupby(col, observed=True)['churn'].mean() * 100
    sns.barplot(x=rate.index.astype(str), y=rate.values, color='#cf222e', ax=a)
    a.axhline(df['churn'].mean() * 100, color='black', ls='--', lw=1)
    a.set_title(f'Churn % by {title}', fontsize=10)
    a.set_xlabel(''); a.set_ylabel('Churn rate (%)')
plt.tight_layout()
plt.savefig('02_churn_by_behaviour.png', dpi=150, bbox_inches='tight')
plt.show()

print(df.groupby('Total_Relationship_Count')['churn'].agg(['mean', 'count']).round(3).T.to_string())
print()
print(df.groupby('Contacts_Count_12_mon')['churn'].agg(['mean', 'count']).round(3).T.to_string())
print()
print(df.groupby('trans_ct_band', observed=True)['churn'].agg(['mean', 'count']).round(3).T.to_string())
print()
print(df.groupby('zero_revolving')['churn'].agg(['mean', 'count']).round(3).T.to_string())

## 5. Churners vs loyal customers: key numbers

In [ ]:
cols = ['Total_Trans_Ct', 'Total_Trans_Amt', 'Total_Ct_Chng_Q4_Q1', 'Total_Amt_Chng_Q4_Q1',
        'Total_Revolving_Bal', 'Avg_Utilization_Ratio']
fig, ax = plt.subplots(2, 3, figsize=(14, 7))
for a, c in zip(ax.flat, cols):
    sns.boxplot(data=df, x='Attrition_Flag', y=c, hue='Attrition_Flag', legend=False,
                palette=['#2da44e', '#cf222e'], ax=a)
    a.set_title(c); a.set_xlabel(''); a.set_ylabel('')
    a.set_xticks([0, 1]); a.set_xticklabels(['Existing', 'Attrited'])
plt.tight_layout()
plt.savefig('03_churners_vs_existing.png', dpi=150, bbox_inches='tight')
plt.show()

summary = df.groupby('Attrition_Flag')[cols + ['Credit_Limit', 'Customer_Age']].mean().T.round(2)
print(summary.to_string())

num = df.select_dtypes('number').drop(columns=['CLIENTNUM'])
corr = num.corr()['churn'].drop('churn').sort_values()
plt.figure(figsize=(7, 5))
sns.barplot(x=corr.values, y=corr.index, color='#8250df')
plt.title('Correlation with churn')
plt.savefig('04_correlation_with_churn.png', dpi=150, bbox_inches='tight')
plt.show()

## 6. Predicting churn

- Stratified 80/20 train/test split, so both sets keep the 16% churn rate.
- One-hot encoding for categorical columns.
- Class imbalance handled with `class_weight='balanced'` (the test set is never resampled).
- Logistic Regression (scaled features) as a simple baseline, and Random Forest as a stronger model.

In [ ]:
X = df.drop(columns=['CLIENTNUM', 'Attrition_Flag', 'churn', 'trans_ct_band', 'zero_revolving', 'Avg_Open_To_Buy'])
X = pd.get_dummies(X, drop_first=True)
y = df['churn']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, stratify=y, random_state=RANDOM_STATE)
print('Train:', X_train.shape, '| Test:', X_test.shape)
print('Churn rate train / test:', round(y_train.mean(), 3), '/', round(y_test.mean(), 3))

models = {
    'Logistic Regression': make_pipeline(
        StandardScaler(),
        LogisticRegression(class_weight='balanced', max_iter=2000, random_state=RANDOM_STATE)),
    'Random Forest': RandomForestClassifier(
        n_estimators=300, class_weight='balanced_subsample',
        n_jobs=-1, random_state=RANDOM_STATE),
}
for name, m in models.items():
    m.fit(X_train, y_train)
    print(name, 'trained')

## 7. Results on the test set
Metrics are for the **churn** class (label 1).

In [ ]:
rows, probs = [], {}
for name, m in models.items():
    pred = m.predict(X_test)
    prob = m.predict_proba(X_test)[:, 1]
    probs[name] = prob
    rows.append({'Model': name,
                 'Accuracy': accuracy_score(y_test, pred),
                 'Precision': precision_score(y_test, pred),
                 'Recall': recall_score(y_test, pred),
                 'F1-score': f1_score(y_test, pred),
                 'ROC-AUC': roc_auc_score(y_test, prob),
                 'PR-AUC': average_precision_score(y_test, prob)})
results = pd.DataFrame(rows).set_index('Model').round(4)
print(results.to_string())
results.to_csv('model_results.csv')

print()
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
for name, m in models.items():
    s = cross_val_score(m, X, y, cv=cv, scoring='roc_auc')
    print(f'{name}: 5-fold ROC-AUC = {s.mean():.3f} (+/- {s.std():.3f})')

In [ ]:
fig, ax = plt.subplots(1, 4, figsize=(19, 4.2))
for a, (name, m) in zip(ax[:2], models.items()):
    cm = confusion_matrix(y_test, m.predict(X_test))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', cbar=False, ax=a,
                xticklabels=['Stay', 'Churn'], yticklabels=['Stay', 'Churn'])
    a.set_title(name); a.set_xlabel('Predicted'); a.set_ylabel('Actual')
for name, p in probs.items():
    fpr, tpr, _ = roc_curve(y_test, p)
    ax[2].plot(fpr, tpr, label=f'{name} (AUC {roc_auc_score(y_test, p):.3f})')
    pr, rc, _ = precision_recall_curve(y_test, p)
    ax[3].plot(rc, pr, label=f'{name} (AP {average_precision_score(y_test, p):.3f})')
ax[2].plot([0, 1], [0, 1], 'k--'); ax[2].set_title('ROC curve')
ax[2].set_xlabel('False positive rate'); ax[2].set_ylabel('True positive rate'); ax[2].legend()
ax[3].set_title('Precision-recall curve'); ax[3].set_xlabel('Recall'); ax[3].set_ylabel('Precision'); ax[3].legend()
plt.tight_layout()
plt.savefig('05_model_evaluation.png', dpi=150, bbox_inches='tight')
plt.show()

for name, m in models.items():
    tn, fp, fn, tp = confusion_matrix(y_test, m.predict(X_test)).ravel()
    print(f'{name}: caught {tp} of {tp + fn} churners, wrongly flagged {fp} loyal customers')

## 8. What drives the predictions?

In [ ]:
rf = models['Random Forest']
imp = pd.Series(rf.feature_importances_, index=X.columns).sort_values(ascending=False).head(12)
plt.figure(figsize=(7, 5))
sns.barplot(x=imp.values[::-1], y=imp.index[::-1], color='#1f6feb')
plt.title('Top 12 features: Random Forest')
plt.savefig('06_feature_importance.png', dpi=150, bbox_inches='tight')
plt.show()
print((imp * 100).round(1).to_string())

## 9. Business view: who should the bank call first?
Rank test customers by predicted churn risk and check how many real churners fall in the highest-risk groups.

In [ ]:
risk = pd.DataFrame({'prob': probs['Random Forest'], 'churn': y_test.values}).sort_values('prob', ascending=False)
n_churn = risk['churn'].sum()
rows = []
for pct in [5, 10, 20, 30]:
    top = risk.head(int(len(risk) * pct / 100))
    rows.append({'Top % of customers contacted': f'{pct}%',
                 'Customers': len(top),
                 'Churners caught': int(top['churn'].sum()),
                 '% of all churners caught': round(100 * top['churn'].sum() / n_churn, 1),
                 'Hit rate (precision)': round(100 * top['churn'].mean(), 1)})
lift = pd.DataFrame(rows)
print(lift.to_string(index=False))
lift.to_csv('risk_ranking.csv', index=False)

## 10. Key insights and recommendations

1. **About 16% of customers churn** (1,627 of 10,127).
2. **Behaviour beats demographics.** Age, gender, income and education barely change the churn rate.
3. **Low and falling activity is the strongest signal.** Churners make far fewer transactions and their activity dropped more between Q1 and Q4.
4. **Few products means high risk.** Customers holding only 1-2 products churn about twice as often as those with 4 or more.
5. **Many contacts with the bank is a warning sign.** Churn rises steadily with the number of contacts (small groups at 5-6 contacts, so treat with care).
6. **Zero revolving balance** customers churn far more, since they use the card less.

**Recommendations**
- Use the model to rank customers by risk each month and contact the top group first (see section 9).
- Trigger a retention offer when a customer's transactions fall sharply compared with the previous quarter.
- Cross-sell a second or third product to single-product customers.
- Review customers who contacted the bank many times; they may have unresolved complaints.

*These are associations from historical data, not proof of cause. Before using the model in practice the bank should test it on new data.*

## 11. Download outputs

In [ ]:
for f in ['model_results.csv', 'risk_ranking.csv', '01_churn_by_demographics.png', '02_churn_by_behaviour.png',
          '03_churners_vs_existing.png', '04_correlation_with_churn.png', '05_model_evaluation.png',
          '06_feature_importance.png']:
    files.download(f)